In [15]:
import pandas as pd
from tqdm import tqdm
import langid
import re
import string
import numpy as np
import inflect
import spacy
from nltk.corpus import stopwords
import contractions
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, save_npz, load_npz
import joblib
import os

In [ ]:
with open("test.ft.txt" , "r" , encoding="utf-8") as file :
    listOfLines = file.readlines()

listOfRatings = []
listOfReviews = []

for line in listOfLines:
    line = line.replace('\n' , '')
    for rating in line.split(" " , 1)[0][-1]:
        listOfRatings.append(rating)
    listOfReviews.append(line.split(" " , 1)[1])
df = pd.DataFrame({"Review" : listOfReviews ,
                   "Rating" : listOfRatings})

df["Review"] = df["Review"].astype("object")
df["Rating"] = df["Rating"].astype("int32")

df.to_csv("testDataFrame.csv")
print(df.Rating.value_counts() , end="\n\n DataFrame shape = ")
print(df.shape)

Rating
2    200000
1    200000
Name: count, dtype: int64

 DataFrame shape = (400000, 2)


In [4]:
df = pd.read_csv("testDataFrame.csv")

def detect_lang(text):
    """Detect the language of the given text."""
    try:
        return langid.classify(text)[0] if isinstance(text, str) and text.strip() else "unknown"
    except:
        return "unknown"

tqdm.pandas(desc="Detecting Language")
df["Language"] = df["Review"].progress_apply(detect_lang)

df.to_csv('DataFrameWithLanguageCol.csv')

print(f"Number of spanish reviews: {3600000 - df.shape[0]}")

Detecting Language: 100%|██████████| 400000/400000 [16:29<00:00, 404.10it/s]


Number of spanish reviews: 3200000


In [2]:
df = pd.read_csv("DataFrameWithLanguageCol.csv", usecols=["Review", "Rating", "Language"])

df = df[df["Language"] == "en"].copy()
df.drop(columns=["Language"], inplace=True)

print(f"DataFrame shape after filtering to English reviews: {df.shape}" , end="\n\n")
print("Value counts for 'Rating' column:")
print(df['Rating'].value_counts(), end="\n\n")
print(f"DataFrame Columns ({len(df.columns)} total):\n- " + "\n- ".join(df.columns) , end="\n\n")
print(f"Duplicate reviews: {df['Review'].duplicated().sum()} / {len(df)} ({df['Review'].duplicated().sum() / len(df):.2%})")

DataFrame shape after filtering to English reviews: (398473, 2)

Value counts for 'Rating' column:
Rating
1    199417
2    199056
Name: count, dtype: int64

DataFrame Columns (2 total):
- Review
- Rating

Duplicate reviews: 0 / 398473 (0.00%)


In [4]:
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
stop_words = set(stopwords.words("english")) - {"not", "no", "nor", "n't"}
p = inflect.engine()

def remove_punctuation_except(text, keep="!?"):
    """Remove all punctuation except specified characters."""
    return "".join(char for char in text if char.isalnum() or char.isspace() or char in keep)

def convert_nums(text, failed_indices=None, current_idx=None):
    """Convert numeric strings to their word equivalents.
    Record the index of text if exception happens."""
    def replacer(match):
        try:
            num_str = match.group()
            if num_str.endswith('.') and num_str.count('.') == 1:
                num_str = num_str.rstrip('.')
            if '.' in num_str:
                int_part, decimal_part = num_str.split('.', 1)
                int_words = p.number_to_words(int_part, andword='') if int_part else ''
                decimal_words = ' '.join(
                    p.number_to_words(d) for d in decimal_part if d.isdigit()
                ) if decimal_part else ''
                return f"{int_words} point {decimal_words}".strip()
            return p.number_to_words(num_str, andword='')
        except Exception:
            print(f"[Warning] Failed to convert number: '{match.group()}'")
            if failed_indices is not None and current_idx is not None:
                failed_indices.add(current_idx)  # record failure index
            return match.group()
    return re.sub(r'\d+(\.\d+)?', replacer, text)

def preprocess_raw_text(text, failed_indices=None, idx=None):
    """Preprocess raw text (expand contractions, remove punctuation except !?, convert nums, normalize whitespace)."""
    text = contractions.fix(text)                                              # Expand contractions
    text = text.strip().lower()                                                # Lowercase and trim
    text = remove_punctuation_except(text, keep="!?")                          # Remove unwanted punctuation
    text = convert_nums(text, failed_indices=failed_indices, current_idx=idx)  # Convert numbers to words
    text = " ".join(text.split())                                              # Normalize whitespace
    return text

def preprocess_batch(texts):
    """Apply full preprocessing and lemmatization to a list of text entries."""
    failed_indices = set()  # store indices of texts that raised exception during number conversion

    # Step 1: preprocess raw texts (expand contractions, punctuation removal, number conversion)
    cleaned_texts = [
        preprocess_raw_text(text, failed_indices=failed_indices, idx=i)
        for i, text in enumerate(tqdm(texts, desc="Preprocessing Raw Texts"))
    ]

    # Step 2: run SpaCy pipeline once on cleaned texts to get lemmas and remove stopwords
    cleaned_lemmas = []
    for doc in tqdm(nlp.pipe(cleaned_texts, batch_size=1000), total=len(cleaned_texts), desc="Lemmatizing"):
        # Lemmatize and remove stopwords (except negations)
        lemmas = [
            token.lemma_ for token in doc
            if token.text not in stop_words and not token.is_punct and not token.is_space
        ]
        cleaned_lemmas.append(" ".join(lemmas))

    return cleaned_lemmas, failed_indices


def addTextFeatures(df, textCol="Review", failedIndices=None, index_map=None):
    """
    Add columns to DataFrame:
      - failedNumberConversion: bool indicating if index is in failedIndices
      - sentenceLength: number of words in the textCol
      - numExclamations: count of '?' and '!' in textCol

    Parameters:
      df (pd.DataFrame): DataFrame to add columns to
      textCol (str): name of the text column
      failedIndices (set or list): indices where number conversion failed (relative to input list)
      index_map (list or Series): maps positions in the failedIndices to actual DataFrame indices

    Returns:
      pd.DataFrame: DataFrame with new columns added
    """
    if failedIndices is None:
        failedIndices = set()

    # Map failed indices to actual df indices if index_map is given
    if index_map is not None:
        actual_failed_indices = [index_map[i] for i in failedIndices]
    else:
        actual_failed_indices = failedIndices

    df["failedNumberConversion"] = 0
    df.loc[actual_failed_indices, "failedNumberConversion"] = 1
    
    # Create new column with sentence length
    df["sentenceLength"] = df[textCol].apply(lambda x: len(str(x).split()))
    
    # Count the number of exclamation and question marks in each original review to capture emphatic or interrogative tone
    df["numExclamations"] = df[textCol].apply(lambda x: sum(x.count(c) for c in ["?", "!"]))
    
    return df

texts = df["Review"].tolist()

# Run preprocessing on full dataset texts
processed_texts, failed_indices = preprocess_batch(texts)
df["processedReview"] = processed_texts

# Create index_map for the full df (list of actual df indices)
index_map = df.index.to_list()

# Add text features with the correct index_map
df = addTextFeatures(
    df,
    textCol="processedReview",
    failedIndices=failed_indices,
    index_map=index_map
)

df.to_csv("preprocessedDataFrame.csv", index=False)
print(f"DataFrame successfully saved.")
print(f"preprocessing done. DataFrame shape: {df.shape}" , end="\n\n")

Preprocessing Raw Texts:  12%|█▏        | 48092/398473 [00:07<00:55, 6346.30it/s]

[Warning] Failed to convert number: '1000000000000000000000000000000000000'


Preprocessing Raw Texts:  15%|█▌        | 59830/398473 [00:09<00:52, 6487.84it/s]

[Warning] Failed to convert number: '4574657467465736734674365874377575456546610'


Preprocessing Raw Texts:  28%|██▊       | 110074/398473 [00:15<00:39, 7369.02it/s]

...

[Warning] Failed to convert number: '124354653677664367434543543655467786545437'


Preprocessing Raw Texts:  82%|████████▏ | 328728/398473 [00:42<00:08, 7989.51it/s]

[Warning] Failed to convert number: '100000000000000000000000000000000000000000000000000000000'


Lemmatizing: 100%|██████████| 398473/398473 [25:20<00:00, 262.10it/s] 


DataFrame successfully saved.
preprocessing done. DataFrame shape: (398473, 6)



In [21]:
df = pd.read_csv("test.ft.txt\preprocessedDataFrame.csv")
df.drop(columns='Review' , inplace=True)
print(f'empty entries = {df["processedReview"].isna().sum()}')
df = df.dropna(subset=["processedReview"])

text = df["processedReview"].tolist()
tfidf = joblib.load(os.path.join(os.getcwd(), "train.ft.txt", "tfidf_vectorizer.joblib"))
tfidf_matrix = tfidf.transform(text)

features_df = df[['Rating', 'failedNumberConversion', 'sentenceLength', 'numExclamations']].reset_index(drop=True)

features_sparse = pd.DataFrame(features_df).astype(float).values

combined_sparse = hstack([tfidf_matrix, features_sparse])

save_npz("TEST_tfidfPlusFeatures.npz", combined_sparse)

column_names = list(tfidf.get_feature_names_out()) + list(features_df.columns)
joblib.dump(column_names, "test.ft.txt\TEST_tfidfPlusFeaturesColumns.pkl")

print("Sparse matrix and column names saved successfully.")

empty entries = 0
Sparse matrix and column names saved successfully.


<h5>Fares Alfares</h5>